# Evaluating trained models on Colab

Scores the checkpoints of a run on Drive with `mammo-seg evaluate`, like
`06_kaggle_evaluation.ipynb` on Kaggle: each fold's model is run on its own
validation fold under a grid of thresholds and minimum region sizes, next to the
baseline of predicting nothing. Research use only: this is not a medical device.

**Once per session:** *Runtime* → *Change runtime type* → *T4 GPU*, before running
anything. Evaluating also works without a GPU if `--device cpu` is added to
`EVALUATE_OPTIONS`, only more slowly. Never put `kaggle.json` or any other
credential in a notebook.

**The data** comes from an archive of the processed data of `DATASET` (`inbreast`
or `cbis-ddsm`), made once on a machine that has it (`mammo-seg manifest`,
`splits`, `masks` and `preprocess`, see the README) and uploaded to `DRIVE_DIR` as
`mammo_seg_data/processed/<DATASET>/mammo_processed_1024.tar.gz` (one folder per
dataset, as in `data/processed/`). It is made with:

```bash
tar czf mammo_processed_1024.tar.gz -C data/processed/<DATASET> \
    manifest.json splits.json preprocessed
```

It holds no raw images, but it is still the dataset: keep the Drive folder private
and never share it (the INbreast license is unclear; CBIS-DDSM must be cited).
CBIS-DDSM has no pixel size, so it is scored with a minimum region size of 0 only
(`--min-areas-mm2 0`) and without areas in mm². The splits are made with a fixed
seed, so an archive of the same manifest has the same folds as the Kaggle notebooks;
the setup prints the hashes of the splits and of the preprocessing to check that. Use the archive the run was trained on: the evaluation stops if the
checkpoints were trained on images preprocessed otherwise.

**The results** are written next to the checkpoints, in the runs folder on Drive:
`evaluation_validation.json` (or `_test`), with `--details` also
`evaluation_validation_details.json`, and `evaluation_code_version.txt`. Download
them into `results/`.

**The test set** is scored by adding `--test` to `EVALUATE_OPTIONS`. Do that once, at
the very end, with the threshold and minimum region size you chose on the validation
folds: choosing a setting on the test set makes the final number optimistic.

In [ ]:
from pathlib import Path

DATASET = "inbreast"  # or "cbis-ddsm"; that of the run
FOLDS = [0, 1, 2, 3, 4]
# More options of `mammo-seg evaluate`, for example "--checkpoint-name best.pt" or
# "--thresholds 0.4 0.5 --min-areas-mm2 0 5". "--test" scores the held-out test set;
# "--details" also writes every image with its masses and false regions.
EVALUATE_OPTIONS = "--details"
DRIVE_DIR = Path("/content/drive/MyDrive/mammo_seg")  # the project folder on Drive
RUN_NAME = "whole-1024"  # the run to evaluate, in DRIVE_DIR / "mammo_seg_runs"
REPO_URL = "https://github.com/GiannisPapa97/mammo_seg.git"
REPO_REF = "main"  # branch or tag to install

## 1. Setup

Mounts Drive, unpacks the archive of `DATASET` into `/content` (once per session), finds the
checkpoints of the run and installs the package. The splits of the archive decide
which images each fold validates on; a checkpoint trained for another fold is
refused.

In [ ]:
import hashlib
import shutil

from google.colab import drive

drive.mount("/content/drive")

WORK = Path("/content")
REPO = WORK / "repo"
assert DATASET in ("inbreast", "cbis-ddsm"), f"unknown DATASET {DATASET!r}"
PROCESSED = WORK / "data" / "processed" / DATASET
MANIFEST = PROCESSED / "manifest.json"
SPLITS = PROCESSED / "splits.json"
PREPROCESSED = PROCESSED / "preprocessed"
RUNS = DRIVE_DIR / "mammo_seg_runs" / RUN_NAME
ARCHIVE = (
    DRIVE_DIR / "mammo_seg_data" / "processed" / DATASET / "mammo_processed_1024.tar.gz"
)
assert ARCHIVE.is_file(), f"upload the archive of the processed data to {ARCHIVE}"

# Unpacked on the machine: reading thousands of files from Drive is slow.
if not (PREPROCESSED / "preprocessing.json").exists():
    PROCESSED.mkdir(parents=True, exist_ok=True)
    local_archive = WORK / ARCHIVE.name
    shutil.copy(ARCHIVE, local_archive)
    !tar xzf "{local_archive}" -C "{PROCESSED}"
for path in (MANIFEST, SPLITS, PREPROCESSED / "preprocessing.json"):
    assert path.exists(), f"{path.relative_to(PROCESSED)} is not in {ARCHIVE.name}"
# Runs that continue or are compared with another platform need the same files.
for path in (SPLITS, PREPROCESSED / "preprocessing.json"):
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()}")

!test -d {REPO} || git clone -q --branch {REPO_REF} {REPO_URL} {REPO}
!pip install -q "{REPO}[cloud]"
!nvidia-smi -L

missing = [fold for fold in FOLDS if not (RUNS / f"fold_{fold}" / "last.pt").exists()]
assert not missing, f"no fold_<k>/last.pt in {RUNS} for the folds {missing}"
test_set = "--test" in EVALUATE_OPTIONS
OUTPUT = RUNS / f"evaluation_{'test' if test_set else 'validation'}.json"

# The code of this session: the time and the commit, next to the evaluation.
VERSION = RUNS / "evaluation_code_version.txt"
!echo "$(date -u +%FT%TZ) $(git -C {REPO} rev-parse HEAD)" | tee -a "{VERSION}"

## 2. Evaluate

The table pools the folds: every image counts once, scored by the model that did not
train on it. The columns are those of `06_kaggle_evaluation.ipynb`: Dice,
sensitivity and precision on the images with a mass, false alarms and false-positive
pixels on the images without, the masses found and the false regions per image, and
the Dice over all images and balanced. `*` marks the best balanced Dice. A model that
predicts nothing scores the share of images without a mass on the all-images Dice,
so that Dice means little unless it is above the `predict nothing` row.

In [ ]:
folds = " ".join(str(fold) for fold in FOLDS)
# CBIS-DDSM has no pixel size, so no minimum region size in mm².
DATASET_OPTIONS = "--min-areas-mm2 0" if DATASET == "cbis-ddsm" else ""
!mammo-seg evaluate --manifest {MANIFEST} --splits {SPLITS} \
    --preprocessed-dir {PREPROCESSED} --runs-dir "{RUNS}" --output "{OUTPUT}" \
    --device cuda --num-workers 2 {DATASET_OPTIONS} {EVALUATE_OPTIONS} --folds {folds}
drive.flush_and_unmount()  # mount again (the setup cell) to use Drive afterwards